# 📊 Projet ML — Prévision du Taux de Change EUR/TND
## Notebook 01 : Analyse Exploratoire des Données (EDA)

**Auteur :** *Ben Selma Hiba &Cherchir Aya*  
**Dataset :** Indicateurs macroéconomiques tunisiens (2015–2025) + données de marché (yfinance)  
**Problématique :** Peut-on prédire le taux de change EUR/TND à partir d'indicateurs macroéconomiques (inflation, PIB, taux d'intérêt, balance commerciale) et de données de marché mondiales ?


---
## 1. Définition du Problème

### Contexte Business
Le taux de change EUR/TND est un indicateur clé pour les entreprises tunisiennes importatrices/exportatrices, les investisseurs et les décideurs économiques. Une prévision fiable de ce taux permet de mieux gérer les risques de change et d'anticiper les mouvements économiques.

### Objectif Technique
Construire un modèle de régression capable de **prédire le taux de change EUR/TND du lendemain** à partir de features macroéconomiques et de marché.

### Variable Cible
- `eurtnd` : taux de change EUR/TND (journalier)

### Sources de Données
| Dataset | Source | Fréquence | Description |
|---------|--------|-----------|-------------|
| Inflation | Fichier local (`inflation.xlsx`) | Mensuelle → Journalière | Taux d'inflation tunisien |
| PIB | Fichier local (`PIB1.xlsx`) | Trimestrielle → Journalière | Produit Intérieur Brut |
| Taux d'intérêt | Fichier local (`taux_interet.xlsx`) | Mensuelle → Journalière | Taux directeur BCT |
| Balance Commerciale | Fichier local (`final_balance_commerciale_data.xlsx`) | Mensuelle → Journalière | Solde commercial |
| EUR/USD, Brent, Or, DXY, etc. | yfinance | Journalière | Données de marché internationales |
| EUR/TND | yfinance (`EURTND=X`) | Journalière | Variable cible |


---
## 2. Préparation & Interpolation des Données Macroéconomiques

Les données macroéconomiques locales sont publiées à fréquence mensuelle ou trimestrielle. Pour les aligner avec les données de marché journalières, nous les **interpolons à la fréquence quotidienne** :
- **Inflation** : interpolation exponentielle (compounding journalier)
- **PIB** : interpolation cubique (données trimestrielles)
- **Taux d'intérêt** : forward-fill (les taux changent par décision, pas continûment)
- **Balance commerciale** : interpolation linéaire


In [ ]:
# ============================================================
# ÉTAPE 2 : Interpolation des données macroéconomiques
# Chaque indicateur passe de fréquence basse → journalière
# ============================================================

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print("=" * 60)
print("2.1 — INFLATION (mensuelle → journalière)")
print("=" * 60)

# --- Charger le fichier brut ---
inflation_raw = pd.read_excel('/content/datasets2/inflation.xlsx')
df_inf = inflation_raw.iloc[:2]

dates_inf = df_inf.iloc[0, 2:].values
inflation_vals = df_inf.iloc[1, 2:].values

month_map = {
    "Jan": "Jan", "Fév": "Feb", "Fev": "Feb", "Mar": "Mar",
    "Avr": "Apr", "Mai": "May", "Jui": "Jun", "Jul": "Jul",
    "Aoû": "Aug", "Aou": "Aug", "Sep": "Sep", "Oct": "Oct",
    "Nov": "Nov", "Déc": "Dec", "Dec": "Dec"
}

clean_dates = []
for d in dates_inf:
    d = str(d)
    for fr, en in month_map.items():
        d = d.replace(fr, en)
    clean_dates.append(d)

inflation_monthly = pd.DataFrame({
    "Date": pd.to_datetime(clean_dates, format="%b %Y"),
    "Inflation": pd.to_numeric(inflation_vals, errors='coerce')
}).set_index("Date").sort_index().dropna()
inflation_monthly = inflation_monthly.iloc[:-1]

# Interpolation exponentielle journalière
inflation_monthly["factor"] = 1 + inflation_monthly["Inflation"] / 100
inflation_monthly["daily_factor"] = inflation_monthly["factor"] ** (1/30) - 1
inflation_daily = inflation_monthly.resample("D").ffill()
inflation_daily["Inflation_daily"] = inflation_daily["daily_factor"] * 100
inflation_daily = inflation_daily[['Inflation_daily']]

print(f"  Lignes mensuelles : {len(inflation_monthly)}")
print(f"  Lignes journalières : {len(inflation_daily)}")
print(f"  Période : {inflation_daily.index.min().date()} → {inflation_daily.index.max().date()}")
print(inflation_daily.head(3))

# Sauvegarder
inflation_daily.reset_index().to_excel("inflation_daily.xlsx", index=False)
print("  Sauvegardé : inflation_daily.xlsx")


In [ ]:
print("=" * 60)
print("2.2 — TAUX D'INTÉRÊT (mensuel → journalier, forward-fill)")
print("=" * 60)

taux_df = pd.read_excel('/content/datasets2/taux_interet.xlsx')
taux_df.columns = taux_df.columns.str.strip()
taux_df.rename(columns={'Date/Indicateurs': 'year'}, inplace=True)

# Filtrer 2015–2025
taux_df = taux_df[(taux_df['year'].dt.year >= 2015) &
                  (taux_df['year'].dt.year <= 2025)]

mois_dict = {
    'Janvier':1, 'Février':2, 'Mars':3, 'Avril':4, 'Mai':5, 'Juin':6,
    'Juillet':7, 'Août':8, 'Septembre':9, 'Octobre':10, 'Novembre':11, 'Décembre':12
}

df_long = taux_df.melt(
    id_vars=['year'],
    value_vars=list(mois_dict.keys()),
    var_name='month', value_name='taux_interet'
)
df_long['month_num'] = df_long['month'].map(mois_dict)
df_long['date'] = pd.to_datetime(dict(
    year=df_long['year'].dt.year,
    month=df_long['month_num'], day=1
))
df_long = df_long[['date', 'taux_interet']].sort_values('date').reset_index(drop=True)

# Forward-fill journalier (le taux reste constant entre les décisions)
taux_daily = df_long.set_index('date').resample('D').ffill().reset_index()
taux_daily = taux_daily[
    (taux_daily['date'] >= '2015-01-01') & (taux_daily['date'] <= '2026-01-01')
]

print(f"  Lignes journalières : {len(taux_daily)}")
print(f"  Période : {taux_daily['date'].min().date()} → {taux_daily['date'].max().date()}")
print(taux_daily.head(3))

taux_daily.to_excel('taux_interet_daily.xlsx', index=False)
print("  Sauvegardé : taux_interet_daily.xlsx")


In [ ]:
print("=" * 60)
print("2.3 — PIB (trimestriel → journalier, interpolation cubique)")
print("=" * 60)

pib_raw = pd.read_excel('/content/datasets2/PIB1.xlsx')
pib_df = pib_raw.iloc[[1, 30]]

dates_row = pib_df.iloc[0, 1:]
pib_row = pib_df.iloc[1, 1:]

df_pib = pd.DataFrame({'Date': dates_row.values, 'PIB': pib_row.values})
df_pib['PIB'] = pd.to_numeric(df_pib['PIB'], errors='coerce')
df_pib['Year'] = df_pib['Date'].str.extract(r'(\d{4})').astype(int)
df_pib = df_pib[(df_pib['Year'] >= 2015) & (df_pib['Year'] <= 2025)].copy()
df_pib = df_pib.drop(columns=['Year'])

quarter_map = {'I': '01', 'II': '04', 'III': '07', 'IV': '10'}
df_pib['Quarter'] = df_pib['Date'].str.split().str[0]
df_pib['Year_str'] = df_pib['Date'].str.extract(r'(\d{4})')
df_pib['Date'] = pd.to_datetime(
    df_pib['Year_str'] + '-' + df_pib['Quarter'].map(quarter_map) + '-01'
)
df_pib = df_pib.drop(columns=['Quarter', 'Year_str']).set_index('Date')

daily_index = pd.date_range(start=df_pib.index.min(), end=df_pib.index.max(), freq='D')
pib_daily = df_pib.reindex(daily_index).interpolate(method='cubic')

print(f"  Trimestres : {len(df_pib)}")
print(f"  Lignes journalières : {len(pib_daily)}")
print(f"  Période : {pib_daily.index.min().date()} → {pib_daily.index.max().date()}")
print(pib_daily.head(3))

pib_daily.reset_index().rename(columns={'index': 'date'}).to_excel('PIB_daily.xlsx', index=False)
print("  Sauvegardé : PIB_daily.xlsx")


In [ ]:
print("=" * 60)
print("2.4 — BALANCE COMMERCIALE (mensuelle → journalière, linéaire)")
print("=" * 60)

bc_df = pd.read_excel('/content/datasets2/final_balance_commerciale_data.xlsx')
bc_df['date'] = pd.to_datetime(bc_df['date'])
bc_df = bc_df[(bc_df['date'] >= '2015-01-01') & (bc_df['date'] <= '2026-01-01')]
bc_df = bc_df.set_index('date')

bc_daily = bc_df.resample('D').interpolate(method='linear')

print(f"  Lignes journalières : {len(bc_daily)}")
print(f"  Période : {bc_daily.index.min().date()} → {bc_daily.index.max().date()}")
print(bc_daily.head(3))

bc_daily.reset_index().to_excel('balance_commerciale_daily.xlsx', index=False)
print("  Sauvegardé : balance_commerciale_daily.xlsx")


---
## 3. Analyse Exploratoire des Données (EDA)

### 3.1 Chargement du dataset macroéconomique fusionné


In [ ]:
# ============================================================
# ÉTAPE 3 : EDA — Chargement et aperçu global
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Style général
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': '#F8F9FA',
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11
})

# Charger le dataset macro principal (déjà fusionné après interpolation)
macro = pd.read_excel("macro_merged.xlsx", sheet_name="macro_merged", parse_dates=["Date"])
macro = macro.rename(columns={"Date": "date"}).set_index("date")
macro.index = pd.to_datetime(macro.index).tz_localize(None)

print("=" * 60)
print("APERÇU DU DATASET MACROÉCONOMIQUE")
print("=" * 60)
print(f"Shape       : {macro.shape}")
print(f"Période     : {macro.index.min().date()} → {macro.index.max().date()}")
print(f"Colonnes    : {list(macro.columns)}")
print()
print(macro.head(5))


### 3.2 Statistiques Descriptives

In [ ]:
# Statistiques descriptives
print("\n=== STATISTIQUES DESCRIPTIVES ===")
desc = macro.describe().T
desc['missing'] = macro.isna().sum()
desc['missing_%'] = (macro.isna().sum() / len(macro) * 100).round(2)
print(desc.round(3).to_string())


### 3.3 Valeurs Manquantes

In [ ]:
# Visualisation des valeurs manquantes
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Barplot des NaN
missing = macro.isna().sum()
if missing.sum() > 0:
    missing[missing > 0].plot(kind='bar', ax=axes[0], color='#E8593C')
    axes[0].set_title('Nombre de valeurs manquantes par colonne')
    axes[0].set_ylabel('Count')
    axes[0].tick_params(axis='x', rotation=45)
else:
    axes[0].text(0.5, 0.5, 'Aucune valeur manquante', ha='center', va='center',
                 fontsize=14, transform=axes[0].transAxes)
    axes[0].set_title('Valeurs manquantes')

# Heatmap NaN
sns.heatmap(macro.isna(), cbar=False, yticklabels=False, ax=axes[1],
            cmap=['#2ECC71', '#E8593C'])
axes[1].set_title('Carte des valeurs manquantes (rouge = NaN)')

plt.tight_layout()
plt.savefig('eda_missing.png', dpi=120, bbox_inches='tight')
plt.show()
print(f"Total NaN : {macro.isna().sum().sum()}")


### 3.4 Distributions des Variables

In [ ]:
# Distributions de toutes les variables numériques
cols = macro.select_dtypes(include=np.number).columns.tolist()
n_cols = len(cols)
ncols = 3
nrows = (n_cols + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(15, 4 * nrows))
axes = axes.flatten()

for i, col in enumerate(cols):
    axes[i].hist(macro[col].dropna(), bins=40, color='#1F3864', alpha=0.7, edgecolor='white')
    axes[i].set_title(col)
    axes[i].set_xlabel('Valeur')
    axes[i].set_ylabel('Fréquence')

# Cacher les axes vides
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Distributions des Variables Macroéconomiques', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('eda_distributions.png', dpi=120, bbox_inches='tight')
plt.show()


### 3.5 Évolution Temporelle des Indicateurs

In [ ]:
# Visualisation temporelle
fig, axes = plt.subplots(len(cols), 1, figsize=(14, 4 * len(cols)))

for i, col in enumerate(cols):
    ax = axes[i] if len(cols) > 1 else axes
    macro[col].plot(ax=ax, color='#1F3864', linewidth=1.2)
    ax.set_title(f'Évolution temporelle — {col}')
    ax.set_ylabel(col)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

plt.suptitle('Séries temporelles des indicateurs macroéconomiques', fontsize=14, y=1.005)
plt.tight_layout()
plt.savefig('eda_timeseries.png', dpi=120, bbox_inches='tight')
plt.show()


### 3.6 Matrice de Corrélation

In [ ]:
# Matrice de corrélation
corr_matrix = macro.corr()

plt.figure(figsize=(12, 9))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=True,
    fmt=".2f",
    cmap='RdYlBu_r',
    center=0,
    vmin=-1, vmax=1,
    square=True,
    linewidths=0.5,
    cbar_kws={'shrink': 0.8}
)
plt.title('Matrice de Corrélation — Indicateurs Macroéconomiques', fontsize=14)
plt.tight_layout()
plt.savefig('eda_correlation.png', dpi=120, bbox_inches='tight')
plt.show()

# Top corrélations
print("\nTop corrélations absolues :")
corr_flat = corr_matrix.abs().unstack().sort_values(ascending=False)
corr_flat = corr_flat[corr_flat < 1.0].drop_duplicates()
print(corr_flat.head(15).to_string())


### 3.7 Détection des Outliers (Boxplots)

In [ ]:
# Boxplots pour détecter les outliers
fig, axes = plt.subplots(nrows, ncols, figsize=(15, 4 * nrows))
axes = axes.flatten()

for i, col in enumerate(cols):
    axes[i].boxplot(macro[col].dropna(), vert=True, patch_artist=True,
                    boxprops=dict(facecolor='#1F3864', alpha=0.7),
                    medianprops=dict(color='#E8593C', linewidth=2))
    axes[i].set_title(col)
    axes[i].set_ylabel('Valeur')

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Détection des Outliers (Boxplots)', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('eda_boxplots.png', dpi=120, bbox_inches='tight')
plt.show()

# Quantification des outliers par IQR
print("\nOutliers détectés (méthode IQR):")
for col in cols:
    Q1 = macro[col].quantile(0.25)
    Q3 = macro[col].quantile(0.75)
    IQR = Q3 - Q1
    n_out = ((macro[col] < Q1 - 1.5*IQR) | (macro[col] > Q3 + 1.5*IQR)).sum()
    print(f"  {col:30s} : {n_out} outliers ({n_out/len(macro)*100:.1f}%)")


---
## 4. Acquisition des Données de Marché & Fusion Finale


In [ ]:
# ============================================================
# ÉTAPE 4 : Téléchargement des données de marché (yfinance)
# et fusion avec les indicateurs macroéconomiques
# ============================================================

import yfinance as yf

tickers = {
    "EURUSD=X":  "eur_usd",
    "BZ=F":      "brent_oil",
    "GC=F":      "gold",
    "DXY":       "dxy",
    "EURGBP=X":  "eur_gbp",
    "^TNX":      "us_10y_yield",
    "EEM":       "msci_em",
}

START, END = "2015-01-01", "2025-01-02"
print("Téléchargement des données de marché...")
frames = []

for ticker, name in tickers.items():
    try:
        raw = yf.download(ticker, start=START, end=END, auto_adjust=True, progress=False)
        if raw is None or raw.empty:
            print(f"  ⚠ {name} — aucune donnée")
            continue
        if isinstance(raw.columns, pd.MultiIndex):
            raw.columns = raw.columns.get_level_values(0)
        if "Close" not in raw.columns:
            print(f"  ⚠ {name} — colonne Close absente")
            continue
        series = raw["Close"].copy()
        series.name = name
        series.index = pd.to_datetime(series.index).tz_localize(None)
        frames.append(series)
        print(f"  ✓ {name:20s} {len(series)} lignes | {series.index.min().date()} → {series.index.max().date()}")
    except Exception as e:
        print(f"  ✗ {name}: {e}")

market = pd.concat(frames, axis=1)
market_daily = market.reindex(macro.index).ffill(limit=3)

full = macro.join(market_daily, how="inner").dropna()

print(f"\n{'='*50}")
print(f"DATASET FINAL : {full.shape}")
print(f"Colonnes : {list(full.columns)}")
print(f"Période  : {full.index.min().date()} → {full.index.max().date()}")
print(f"{'='*50}")

full.to_csv("macro_market_merged.csv")
print("Sauvegardé : macro_market_merged.csv")


### 4.1 Visualisation du dataset fusionné

In [ ]:
# Visualisation des variables de marché vs macroéconomiques
fig, axes = plt.subplots(3, 1, figsize=(14, 11))

full["eur_usd"].plot(ax=axes[0], color="#1F3864", linewidth=1.2)
axes[0].set_title("EUR/USD — Taux de change")
axes[0].set_ylabel("EUR/USD")

full[["brent_oil", "gold"]].plot(ax=axes[1], secondary_y="gold")
axes[1].set_title("Brent Oil & Gold")

full.select_dtypes(include=np.number).corr().iloc[:, 0].sort_values(ascending=False)[1:8].plot(
    kind='barh', ax=axes[2], color='#1F3864'
)
axes[2].set_title("Corrélations des features avec la 1ère variable numérique")

plt.tight_layout()
plt.savefig("eda_market_overview.png", dpi=120, bbox_inches='tight')
plt.show()


---
## 5. Résumé EDA

| Dimension | Résultat |
|-----------|----------|
| Période | 2015-01-01 → 2025-01-01 |
| Fréquence | Journalière |
| Variables macro | Inflation, PIB, Taux d'intérêt, Balance commerciale |
| Variables marché | EUR/USD, Brent, Or, DXY, EUR/GBP, US 10Y Yield, MSCI EM |
| Valeurs manquantes | Traitées par interpolation ou forward-fill |
| Outliers | Présents sur la période 2020–2022 (COVID + crise TND) |

### Observations clés
- Le taux EUR/TND montre une **tendance haussière continue** depuis 2015, avec une accélération post-2022.
- **EUR/USD** et **Brent Oil** sont parmi les features les plus corrélées avec EUR/TND.
- Les données macroéconomiques ont des **fréquences différentes** → interpolation nécessaire avant modélisation.
- La crise COVID (2020) et la dépréciation du TND (2022) créent des **régimes distincts** à considérer.

> ➡️ Notebook suivant : `02_Modeling.ipynb` — Préprocessing, Feature Engineering et Modélisation
